## Problem Statement

### Business Context

Business communities in the United States are facing high demand for human resources, but one of the constant challenges is identifying and attracting the right talent, which is perhaps the most important element in remaining competitive. Companies in the United States look for hard-working, talented, and qualified individuals both locally as well as abroad.

The Immigration and Nationality Act (INA) of the US permits foreign workers to come to the United States to work on either a temporary or permanent basis. The act also protects US workers against adverse impacts on their wages or working conditions by ensuring US employers' compliance with statutory requirements when they hire foreign workers to fill workforce shortages. The immigration programs are administered by the Office of Foreign Labor Certification (OFLC).

OFLC processes job certification applications for employers seeking to bring foreign workers into the United States and grants certifications in those cases where employers can demonstrate that there are not sufficient US workers available to perform the work at wages that meet or exceed the wage paid for the occupation in the area of intended employment.

### Objective

In FY 2016, the OFLC processed 775,979 employer applications for 1,699,957 positions for temporary and permanent labor certifications. This was a nine percent increase in the overall number of processed applications from the previous year. The process of reviewing every case is becoming a tedious task as the number of applicants is increasing every year.

The increasing number of applicants every year calls for a Machine Learning based solution that can help in shortlisting the candidates having higher chances of VISA approval. OFLC has hired the firm EasyVisa for data-driven solutions. You as a data  scientist at EasyVisa have to analyze the data provided and, with the help of a classification model:

* Facilitate the process of visa approvals.
* Recommend a suitable profile for the applicants for whom the visa should be certified or denied based on the drivers that significantly influence the case status.

### Data Description

The data contains the different attributes of employee and the employer. The detailed data dictionary is given below.

* case_id: ID of each visa application
* continent: Information of continent the employee
* education_of_employee: Information of education of the employee
* has_job_experience: Does the employee has any job experience? Y= Yes; N = No
* requires_job_training: Does the employee require any job training? Y = Yes; N = No
* no_of_employees: Number of employees in the employer's company
* yr_of_estab: Year in which the employer's company was established
* region_of_employment: Information of foreign worker's intended region of employment in the US.
* prevailing_wage:  Average wage paid to similarly employed workers in a specific occupation in the area of intended employment. The purpose of the prevailing wage is to ensure that the foreign worker is not underpaid compared to other workers offering the same or similar service in the same area of employment.
* unit_of_wage: Unit of prevailing wage. Values include Hourly, Weekly, Monthly, and Yearly.
* full_time_position: Is the position of work full-time? Y = Full Time Position; N = Part Time Position
* case_status:  Flag indicating if the Visa was certified or denied

## Installing and Importing the necessary libraries

In [ ]:
# Installing the libraries with the specified version.
!pip install numpy==1.26.0 pandas==1.5.3 scikit-learn==1.5.2 matplotlib==3.7.1 seaborn==0.13.1 xgboost==2.0.3 -q --user

**Note**: *After running the above cell, kindly restart the notebook kernel and run all cells sequentially from the below.*

In [ ]:
# import libraries for data manipulation
import numpy as np
import pandas as pd

# import libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# libraries for building models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KDTree
from sklearn import tree

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    BaggingClassifier,
)

# To oversample and undersample data
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# To do hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# library for tuning different models
from sklearn.model_selection import GridSearchCV

# library to get metric scores and plot the tree
from sklearn import metrics, tree

# Library to split data
from sklearn.model_selection import train_test_split, cross_val_score, KFold

# To get diferent metric scores
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
)


import shap

# To ignore unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

## Import Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv('/content/drive/My Drive/Python_Course/Projects/ML_Project3/EasyVisa.csv')

## Overview of the Dataset

#### View the first and last 5 rows of the dataset

In [ ]:
data.head()

In [ ]:
data.tail()

#### Understand the shape of the dataset

In [ ]:
#Finding shape of Data
print("Rows:", {data.shape[0]})
print("Columns:", {data.shape[1]})

#### Check the data types of the columns for the dataset

In [ ]:
data.describe().T

In [ ]:
df=data.copy()

In [ ]:
df.info()

## Exploratory Data Analysis (EDA)

#### Let's check the statistical summary of the data

In [ ]:
df.describe().T

#### Fixing the negative values in number of employees columns

In [ ]:
#Number of employees has a minium values of -26. Negative values in this column should be treated
df.loc[data['no_of_employees']<=0]

In [ ]:
df['no_of_employees'] = abs(df['no_of_employees'])

In [ ]:
#Output shows that all the data points with number of employees as negative have been fixed.

df.loc[data['no_of_employees']<=0]

#### Let's check the count of each unique category in each of the categorical variables

In [ ]:
#Converting the coilumns of type, object to category
cols = df.select_dtypes(['object'])

for i in cols.columns:
    df[i] = df[i].astype('category')

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

# **Observations:**

There are 25480 unique values for case id column.

6 unique continents

4 unique types of education

2 unique values for job experience

5 inique regions of employment

4 unique units of wage

2 unique values for whether the positiob is full time or not

2 unique values for case status




### Univariate Analysis

In [ ]:
def histogram_boxplot(data, feature, figsize=(15, 10), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (15,10))
    kde: whether to show the density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a triangle will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

#### Observations on education of employee

In [ ]:
labeled_barplot(df, 'education_of_employee',True)

# **Observations:**

Most candidates have a Bachelor's degree - 40.2%

Followed by Master's degree - 37.8% of the candidates

#### Observations on region of employment

In [ ]:
labeled_barplot(df, 'region_of_employment',True)

# **Observations:**

Majority of the candidates request jobs from the Northeast, South and West regions

The Island region has the least number of candidate requests

#### Observations on job experience

In [ ]:
labeled_barplot(df, 'has_job_experience',True)

# **Observations:**

58.1% of candidates have prior job experience

#### Observations on case status

In [ ]:
labeled_barplot(df, 'case_status',True)

# **Observations:**

The dataset seems to be decently balanced considering the distribution of case status among candidates.

In [ ]:
labeled_barplot(df, 'continent',True)

# **Observations:**

Majority of the candidates applying for visa are from Asia, followed by Europe and North America

South America, Africa and Oceania have very low candidates looking for a visa

In [ ]:
labeled_barplot(df, 'requires_job_training', True)

# **Observations:**

A vast majority of the candidates DO NOT require job training - 88.4%

In [ ]:
histogram_boxplot(df, 'no_of_employees')

# **Observations:**

The plots are right skewed

Most companies have less than 10000 employees.

In [ ]:
df_number_of_employees = df[df["no_of_employees"] <= 10000].copy()
histogram_boxplot(df_number_of_employees, 'no_of_employees')

In [ ]:
df_number_of_employees = df[df["no_of_employees"] <= 6000].copy()
histogram_boxplot(df_number_of_employees, 'no_of_employees')

# **Observations:**

Companies with total number of employees between 0-6000 mostly are requesting for a visa

The average number of employees in a company is around 2200

In [ ]:
histogram_boxplot(df, 'prevailing_wage')

# **Observations:**

This plot is also heavily right skewed

The mean and median are close together at around 75000.

In [ ]:
labeled_barplot(df, 'unit_of_wage',True)

# **Observations:**

Majority of the wages are yearly wages

In [ ]:
labeled_barplot(df, 'full_time_position', True)

# **Observations:**

89.4% of candidates in dataset are in a full time position

### Bivariate Analysis

**Creating functions that will help us with further analysis.**

In [ ]:
### function to plot distributions wrt target


def distribution_plot_wrt_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = data[target].unique()

    axs[0, 0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
        stat="density",
    )

    axs[0, 1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
        stat="density",
    )

    axs[1, 0].set_title("Boxplot w.r.t target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()

In [ ]:
def stacked_barplot(data, predictor, target):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 5, 5))
    plt.legend(
        loc="lower left", frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
#Correlation map
cols_list = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(12, 7))
sns.heatmap(df[cols_list].corr(), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral");

# **Observations:**

There is minimal correlation between the numeric variables

#### Does higher education increase the chances of visa certification for well-paid jobs abroad?

In [ ]:
stacked_barplot(df, 'education_of_employee', 'case_status')

# **Observations:**

Yes, higher the education, more likely the visa approval - Doctorate degree has the highest approval rate.

Candidates with a High School degree have a 30% chance of approval.

Candidates with a Bachelor's degree or higher have at least a 60% approval rate.

Master's degree and Doctorate degree holders have an 80% and 90% approval rates respectively.

#### How does visa status vary across different continents?

In [ ]:
stacked_barplot(df, 'continent', 'case_status')

# **Observations:**

Approval rates range from 60-80% across all continents

South America has the lowest approval rate with a little below 60%.

Europe has the highest approval rate with ~80% of the visas being certified.

In [ ]:
stacked_barplot(df, 'region_of_employment', 'case_status')

# **Observations:**

From univariate analysis, Midwest and Island have the least amount of candidates applying for. However, the visa approval rates for both these regions are between 60-70%

West,Northeast and South regions also have 60-70% approval rate.

In [ ]:
stacked_barplot(df, 'region_of_employment', 'education_of_employee')

# **Observations:**

Every region has visa requests from all 4 education levels.

Comparatively, workers with the highest level of education (doctorate) request more employment in the West and Northeast.

Most of workers with a masters request Island or Midwest. The regions with least amounts of requests.

Candidates with Bachelor's degree mostly choose West, Northeast or South regions.

Candidates who completed High school also mainly choose Island and Midwest regions.

#### Does having prior work experience influence the chances of visa certification for career opportunities abroad?

In [ ]:
stacked_barplot(df, 'has_job_experience', 'case_status')

# **Observations:**

There is a slightly better chance of visa approval for candidates who have job experience - around 10% more.

#### Is the prevailing wage consistent across all regions of the US?

In [ ]:
distribution_plot_wrt_target(df,'prevailing_wage', 'region_of_employment')

# **Observations:**

Prevailing wage is changing with the region of employment

Median prevailing wage is highest for the Midwest and Island regions

#### Does visa status vary with changes in the prevailing wage set to protect both local talent and foreign workers?

In [ ]:
distribution_plot_wrt_target(df, 'prevailing_wage', 'case_status')

# **Observations:**

There isn't much difference between accepted and denied case status in relation to prevailing wage.


In [ ]:
distribution_plot_wrt_target(df, 'prevailing_wage', 'education_of_employee')

# **Observations:**

Candidates with a doctorate have a lower median than those with a High School degree.

Master's and Bachelor's candidates have a very similar prevailing wage.

Candidates with a High School degree have a slightly lower prevailing wage than Master's and Bachelor's.

In [ ]:
distribution_plot_wrt_target(df, 'prevailing_wage', 'has_job_experience')

# **Observations:**

The median prevailing wage seems to be almost similar among candidates with job experience and without job experience.

In [ ]:
distribution_plot_wrt_target(df, 'prevailing_wage', 'requires_job_training')

# **Observations:**

Workers that require job training have a slightly higher prevailing wage median.

In [ ]:
sns.catplot(data=df, x='education_of_employee', y='prevailing_wage', hue='has_job_experience', kind='point');

# **Observations:**

Candidates with no experience have higher wages when they have High School and Master's degrees.

Candidates with Doctorate have the lowest prevailing wages, which is surprising

Candidates with Master's degree have the Highest prevailing wage

In [ ]:
distribution_plot_wrt_target(df,'prevailing_wage', 'unit_of_wage')

#### Does the unit of prevailing wage (Hourly, Weekly, etc.) have any impact on the likelihood of visa application certification?

In [ ]:
stacked_barplot(df, 'unit_of_wage', 'case_status')

# **Observations:**

Workers paid hourly are least likely to get their visa accepted

Yearly wage candidates are most likely to be approved with ~75% approval rate

Monthly and weekly wage candidates have similar accptance rate for their visa at 60%

## Data Pre-processing

In [ ]:
# Finding number of missing rows
df.isnull().sum()

In [ ]:
# Finding if there are any duplicate data points
df.duplicated().sum()

# **Observations:**

There are no missing or duplicate values

In [ ]:
# Dropping the Case ID column
df = df.drop(['case_id'], axis=1)

df.head()

In [ ]:
#Changing Case status datatype from category to numeric(binary)
df['case_status'] = df['case_status'].replace({'Denied': 0, 'Certified': 1})

df.info()


In [ ]:
# Adding a row for number of years since company was operational using the available "Year established" column

df['yr_since_company_started'] = 2017 - df['yr_of_estab']
df.head()

In [ ]:
# Dropping the "Year established" column
df.drop(['yr_of_estab'], axis=1, inplace=True)
df.head()

### Outlier Check

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(15, 12))

for i, variable in enumerate(num_cols):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(df[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

### Data Preparation for modeling

In [ ]:
# Defining independent and dependent variables

X = df.drop(['case_status'], axis=1)
y = df['case_status']


In [ ]:
# Splitting data into training, validation and test set:

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=1, stratify=y_temp
)
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
print("Number of rows in train data =", X_train.shape[0])
print("Number of rows in validation data =", X_val.shape[0])
print("Number of rows in test data =", X_test.shape[0])

In [ ]:
# One hot encoding of Train, Validation and test datasets

#print(X_train.shape, X_val.shape, X_test.shape)
X_train = pd.get_dummies(X_train, drop_first=True)
X_train = X_train.astype(float)

X_val = pd.get_dummies(X_val, drop_first=True)
X_val = X_val.astype(float)

X_test = pd.get_dummies(X_test, drop_first=True)
X_test = X_test.astype(float)

print(X_train.shape, X_val.shape, X_test.shape)

## Model Building

### Model Evaluation Criterion

- Choose the primary metric to evaluate the model on
- Elaborate on the rationale behind choosing the metric

First, let's create functions to calculate different metrics and confusion matrix so that we don't have to use the same code repeatedly for each model.
* The `model_performance_classification_sklearn` function will be used to check the model performance of models.
* The `confusion_matrix_sklearn` function will be used to plot the confusion matrix.

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using sklearn


def model_performance_classification_sklearn(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred)  # to compute Recall
    precision = precision_score(target, pred)  # to compute Precision
    f1 = f1_score(target, pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
def confusion_matrix_sklearn(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

#### Defining scorer to be used for cross-validation and hyperparameter tuning

**We are now done with pre-processing and evaluation criterion, so let's start building the model.**

# **Observations:**


For this particular use case, we want to keep the False Positives and the False Negatives at a reasonable level - We do not want to accept visas with have to be rejected NOR reject visas which should have been accepted.

Hence, both Recall and Precision are important.

So, I would consider using F1 score as the defining criteria when building and selecting my models.

### Model building with Original data

In [ ]:
models = []  # Empty list to store all the models

# Appending models into the list
models.append(("Bagging", BaggingClassifier(estimator=DecisionTreeClassifier(random_state=1, class_weight='balanced'), random_state=1)))
models.append(("Random forest", RandomForestClassifier(random_state=1, class_weight='balanced')))
models.append(("GBM", GradientBoostingClassifier(random_state=1)))
models.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models.append(("dtree", DecisionTreeClassifier(random_state=1, class_weight='balanced')))

# Fitting the model to Original dataset and finding the F1 scores for training and validation sets

print("\nTraining Performance:\n")
for name, model in models:
    model.fit(X_train, y_train)
    scores = f1_score(y_train, model.predict(X_train))
    print("{}: {}".format(name, scores))

print("\nValidation Performance:\n")
for name, model in models:
    model.fit(X_train, y_train)
    scores_val = f1_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores_val))

In [ ]:
print("\nTraining and Validation Performance Difference:\n")

for name, model in models:
    model.fit(X_train, y_train)
    scores_train = recall_score(y_train, model.predict(X_train))
    scores_val = recall_score(y_val, model.predict(X_val))
    difference1 = scores_train - scores_val
    print("{}: Training Score: {:.4f}, Validation Score: {:.4f}, Difference: {:.4f}".format(name, scores_train, scores_val, difference1))

# **Observations:**

GBM has the best performance followed by Adaboost and Random Forest

### Model Building with Oversampled data

In [ ]:
print("Before Oversampling, counts of label 'Yes': {}".format(sum(y_train == 1)))
print("Before Oversampling, counts of label 'No': {} \n".format(sum(y_train == 0)))

# Oversampling dataset using SMOTE

sm = SMOTE(
    sampling_strategy=1, k_neighbors=5, random_state=1
)  # Oversampling minority class
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)


print("After Oversampling, counts of label 'Yes': {}".format(sum(y_train_over == 1)))
print("After Oversampling, counts of label 'No': {} \n".format(sum(y_train_over == 0)))


print("After Oversampling, the shape of train_X: {}".format(X_train_over.shape))
print("After Oversampling, the shape of train_y: {} \n".format(y_train_over.shape))

In [ ]:
models = []  # Empty list to store all the models

# Appending models into the list
models.append(("Bagging", BaggingClassifier(estimator=DecisionTreeClassifier(random_state=1, class_weight='balanced'), random_state=1)))
models.append(("Random forest", RandomForestClassifier(random_state=1, class_weight='balanced')))
models.append(("GBM", GradientBoostingClassifier(random_state=1)))
models.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models.append(("dtree", DecisionTreeClassifier(random_state=1, class_weight='balanced')))

# Fitting the model to oversampled data and finding the F1 scores for training and validation sets

print("\n" "Training Performance:" "\n")
for name, model in models:
    model.fit(X_train_over, y_train_over)
    scores = f1_score(y_train_over, model.predict(X_train_over))
    print("{}: {}".format(name, scores))

print("\n" "Validation Performance:" "\n")

for name, model in models:
    model.fit(X_train_over, y_train_over)
    scores = f1_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores))

In [ ]:
print("\nTraining and Validation Performance Difference:\n")

for name, model in models:
    model.fit(X_train_over, y_train_over)
    scores_train = recall_score(y_train_over, model.predict(X_train_over))
    scores_val = recall_score(y_val, model.predict(X_val))
    difference1 = scores_train - scores_val
    print("{}: Training Score: {:.4f}, Validation Score: {:.4f}, Difference: {:.4f}".format(name, scores_train, scores_val, difference1))

# **Observations:**

Adaboost has the best performance followed by GBM and Random Forest.

### Model Building with Undersampled data

In [ ]:
# Undersampling dataset

rus = RandomUnderSampler(random_state=1)
X_train_un, y_train_un = rus.fit_resample(X_train, y_train)

In [ ]:
print("Before Under Sampling, counts of label 'Yes': {}".format(sum(y_train == 1)))
print("Before Under Sampling, counts of label 'No': {} \n".format(sum(y_train == 0)))

print("After Under Sampling, counts of label 'Yes': {}".format(sum(y_train_un == 1)))
print("After Under Sampling, counts of label 'No': {} \n".format(sum(y_train_un == 0)))

print("After Under Sampling, the shape of train_X: {}".format(X_train_un.shape))
print("After Under Sampling, the shape of train_y: {} \n".format(y_train_un.shape))

In [ ]:
models = []  # Empty list to store all the models

# Appending models into the list
models.append(("Bagging", BaggingClassifier(estimator=DecisionTreeClassifier(random_state=1, class_weight='balanced'), random_state=1)))
models.append(("Random forest", RandomForestClassifier(random_state=1, class_weight='balanced')))
models.append(("GBM", GradientBoostingClassifier(random_state=1)))
models.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models.append(("dtree", DecisionTreeClassifier(random_state=1, class_weight='balanced')))

# Fitting the model to the undersampled data and finding the F1 scores for training and validation sets
print("\n" "Training Performance:" "\n")
for name, model in models:
    model.fit(X_train_un, y_train_un)
    scores = f1_score(y_train_un, model.predict(X_train_un))
    print("{}: {}".format(name, scores))

print("\n" "Validation Performance:" "\n")

for name, model in models:
    model.fit(X_train_un, y_train_un)
    scores = f1_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores))

In [ ]:
print("\nTraining and Validation Performance Difference:\n")

for name, model in models:
    model.fit(X_train_un, y_train_un)
    scores_train = recall_score(y_train_un, model.predict(X_train_un))
    scores_val = recall_score(y_val, model.predict(X_val))
    difference3 = scores_train - scores_val
    print("{}: Training Score: {:.4f}, Validation Score: {:.4f}, Difference: {:.4f}".format(name, scores_train, scores_val, difference3))

# **Observations:**

Adaboost has the best performance followed by GBM

Observations:

Based on all of the above, I will choose 3 models for hypertuning



*   Adaboost
*   GBM
*   Random Forest





## Hyperparameter Tuning

**Best practices for hyperparameter tuning in AdaBoost:**

`n_estimators`:

- Start with a specific number (50 is used in general) and increase in steps: 50, 75, 85, 100

- Use fewer estimators (e.g., 50 to 100) if using complex base learners (like deeper decision trees)

- Use more estimators (e.g., 100 to 150) when learning rate is low (e.g., 0.1 or lower)

- Avoid very high values unless performance keeps improving on validation

`learning_rate`:

- Common values to try: 1.0, 0.5, 0.1, 0.01

- Use 1.0 for faster training, suitable for fewer estimators

- Use 0.1 or 0.01 when using more estimators to improve generalization

- Avoid very small values (< 0.01) unless you plan to use many estimators (e.g., >500) and have sufficient data


---

**Best practices for hyperparameter tuning in Random Forest:**


`n_estimators`:

* Start with a specific number (50 is used in general) and increase in steps: 50, 75, 100, 125
* Higher values generally improve performance but increase training time
* Use 100-150 for large datasets or when variance is high


`min_samples_leaf`:

* Try values like: 1, 2, 4, 5, 10
* Higher values reduce model complexity and help prevent overfitting
* Use 1–2 for low-bias models, higher (like 5 or 10) for more regularized models
* Works well in noisy datasets to smooth predictions


`max_features`:

* Try values: `"sqrt"` (default for classification), `"log2"`, `None`, or float values (e.g., `0.3`, `0.5`)
* `"sqrt"` balances between diversity and performance for classification tasks
* Lower values (e.g., `0.3`) increase tree diversity, reducing overfitting
* Higher values (closer to `1.0`) may capture more interactions but risk overfitting


`max_samples` (for bootstrap sampling):

* Try float values between `0.5` to `1.0` or fixed integers
* Use `0.6–0.9` to introduce randomness and reduce overfitting
* Smaller values increase diversity between trees, improving generalization

---

**Best practices for hyperparameter tuning in Gradient Boosting:**

`n_estimators`:

* Start with 100 (default) and increase: 100, 200, 300, 500
* Typically, higher values lead to better performance, but they also increase training time
* Use 200–500 for larger datasets or complex problems
* Monitor validation performance to avoid overfitting, as too many estimators can degrade generalization


`learning_rate`:

* Common values to try: 0.1, 0.05, 0.01, 0.005
* Use lower values (e.g., 0.01 or 0.005) if you are using many estimators (e.g., > 200)
* Higher learning rates (e.g., 0.1) can be used with fewer estimators for faster convergence
* Always balance the learning rate with `n_estimators` to prevent overfitting or underfitting


`subsample`:

* Common values: 0.7, 0.8, 0.9, 1.0
* Use a value between `0.7` and `0.9` for improved generalization by introducing randomness
* `1.0` uses the full dataset for each boosting round, potentially leading to overfitting
* Reducing `subsample` can help reduce overfitting, especially in smaller datasets


`max_features`:

* Common values: `"sqrt"`, `"log2"`, or float (e.g., `0.3`, `0.5`)
* `"sqrt"` (default) works well for classification tasks
* Lower values (e.g., `0.3`) help reduce overfitting by limiting the number of features considered at each split

---

**Best practices for hyperparameter tuning in XGBoost:**

`n_estimators`:

* Start with 50 and increase in steps: 50,75,100,125.
* Use more estimators (e.g., 150-250) when using lower learning rates
* Monitor validation performance
* High values improve learning but increase training time

`subsample`:

* Common values: 0.5, 0.7, 0.8, 1.0
* Use `0.7–0.9` to introduce randomness and reduce overfitting
* `1.0` uses the full dataset in each boosting round; may overfit on small datasets
* Values < 0.5 are rarely useful unless dataset is very large

`gamma`:

* Try values: 0 (default), 1, 3, 5, 8
* Controls minimum loss reduction needed for a split
* Higher values make the algorithm more conservative (i.e., fewer splits)
* Use values > 0 to regularize and reduce overfitting, especially on noisy data


`colsample_bytree`:

* Try values: 0.3, 0.5, 0.7, 1.0
* Fraction of features sampled per tree
* Lower values (e.g., 0.3 or 0.5) increase randomness and improve generalization
* Use `1.0` when you want all features considered for every tree


`colsample_bylevel`:

* Try values: 0.3, 0.5, 0.7, 1.0
* Fraction of features sampled at each tree level (i.e., per split depth)
* Lower values help in regularization and reducing overfitting
* Often used in combination with `colsample_bytree` for fine control over feature sampling

---

In [ ]:
# Tuning AdaBoost model with Original Data

%%time

# defining model
Model = AdaBoostClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(10, 40, 10),
    "learning_rate": [0.1, 0.01, 0.2, 0.05, 1],
    "estimator": [
        DecisionTreeClassifier(max_depth=1, random_state=1),
        DecisionTreeClassifier(max_depth=2, random_state=1),
        DecisionTreeClassifier(max_depth=3, random_state=1),
    ],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_jobs = -1, n_iter=50, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
tuned_adb_orig = AdaBoostClassifier(
    random_state=1,
    n_estimators=20,
    learning_rate=0.1,
    estimator=DecisionTreeClassifier(max_depth=1, random_state=1),
)
tuned_adb_orig.fit(X_train, y_train)

In [ ]:
# Checking model's performance on training set
adb_train_orig = model_performance_classification_sklearn(tuned_adb_orig, X_train, y_train)
adb_train_orig

In [ ]:
# Checking model's performance on validation set
adb_val_orig = model_performance_classification_sklearn(tuned_adb_orig, X_val, y_val)
adb_val_orig

In [ ]:
# Tuning AdaBoost model with Oversampled Data

%%time

# defining model
Model = AdaBoostClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(10, 40, 10),
    "learning_rate": [0.1, 0.01, 0.2, 0.05, 1],
    "estimator": [
        DecisionTreeClassifier(max_depth=1, random_state=1),
        DecisionTreeClassifier(max_depth=2, random_state=1),
        DecisionTreeClassifier(max_depth=3, random_state=1),
    ],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_jobs = -1, n_iter=50, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_over, y_train_over)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
tuned_adb_over = AdaBoostClassifier(
    random_state=1,
    n_estimators=10,
    learning_rate=0.01,
    estimator=DecisionTreeClassifier(max_depth=1, random_state=1),
)
tuned_adb_over.fit(X_train_over, y_train_over)

In [ ]:
# Checking model's performance on training set
adb_train_over = model_performance_classification_sklearn(tuned_adb_over, X_train_over, y_train_over)
adb_train_over

In [ ]:
# Checking model's performance on validation set
adb_val_over = model_performance_classification_sklearn(tuned_adb_over, X_val, y_val)
adb_val_over

# **Observations:**

Recall and F1 score are better on model using Tuned Original Data

Precision and Accuracy are better on model with Tuned Oversampled data

There is a considerable difference in Training and Validation performance for the model with Tuned Oversampled data

Difference in Training and validation performance is negligible for the model with tuned Original data - which is a good indication for selection and improves chances of test data to perform similarly as the training and validation data.

CV score is also higher for model with tuned Original data

In [ ]:
# Tuning Random Forest model with Original Data

%%time

# defining model
Model = RandomForestClassifier(random_state=1)


# Define the hyperparameter search space - Considering sqrt and log2 for max_features since this is a classification problem
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start=100, stop=500, num=10)],
    'max_features': ['sqrt', 'log2', None],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

#Calling RandomizedSearchCV
randomized_cv_rf_orig = RandomizedSearchCV(estimator=Model, param_distributions=param_dist, n_jobs = -1, n_iter=50, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv_rf_orig.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
best_params = randomized_cv_rf_orig.best_params_

# Create a new model with the best parameters
tuned_rf_model = RandomForestClassifier(random_state=1, **best_params)

tuned_rf_model.fit(X_train, y_train)

In [ ]:
# Checking model's performance on training set
rf_train_orig = model_performance_classification_sklearn(tuned_rf_model, X_train, y_train)
rf_train_orig

In [ ]:
# Checking model's performance on Validation set
rf_val_orig = model_performance_classification_sklearn(tuned_rf_model, X_val, y_val)
rf_val_orig

In [ ]:
# Tuning Random Forest model with oversampled data

%%time

# defining model
Model = RandomForestClassifier(random_state=1)

param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start=100, stop=500, num=10)],
    'max_features': ['sqrt', 'log2', None],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

#Calling RandomizedSearchCV
randomized_cv_rf_over = RandomizedSearchCV(estimator=Model, param_distributions=param_dist, n_jobs = -1, n_iter=50, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv_rf_over.fit(X_train_over, y_train_over)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
best_params = randomized_cv_rf_over.best_params_

# Create a new model with the best parameters
tuned_rf_model_over = RandomForestClassifier(random_state=1, **best_params)

tuned_rf_model_over.fit(X_train_over, y_train_over)

In [ ]:
rf_train_over = model_performance_classification_sklearn(tuned_rf_model_over, X_train_over, y_train_over)
rf_train_over

In [ ]:
# Checking model's performance on Validation set
rf_val_over = model_performance_classification_sklearn(tuned_rf_model_over, X_val, y_val)
rf_val_over

In [ ]:
# Tuning Gradient Boosting Model with Original data

%%time

#defining model
Model = GradientBoostingClassifier(random_state=1)

#Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(75, 150, 25),
    "learning_rate": [0.1, 0.01, 0.2, 0.05, 1],
    "subsample": [0.5, 0.7, 1],
    "max_features": [0.5, 0.7, 1],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

#Calling RandomizedSearchCV
randomized_cv_gbm_orig = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=50, scoring=scorer, cv=5, random_state=1, n_jobs = -1)

#Fitting parameters in RandomizedSearchCV
randomized_cv_gbm_orig.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv_gbm_orig.best_params_,randomized_cv_gbm_orig.best_score_))

In [ ]:
tuned_gbm1_Orig = GradientBoostingClassifier(
    random_state=1,
    subsample=0.5,
    n_estimators=100,
    max_features=1,
    learning_rate=0.01,
)
tuned_gbm1_Orig.fit(X_train, y_train)

In [ ]:
# Checking model's performance on training set
gbm1_train_orig = model_performance_classification_sklearn(tuned_gbm1_Orig, X_train, y_train)
gbm1_train_orig

In [ ]:
# Checking model's performance on validation set
gbm1_val_orig = model_performance_classification_sklearn(tuned_gbm1_Orig, X_val, y_val)
gbm1_val_orig

In [ ]:
# tuning Gradient Boosting Model with Ovesampled data

%%time

#defining model
Model = GradientBoostingClassifier(random_state=1)

#Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(75, 150, 25),
    "learning_rate": [0.1, 0.01, 0.2, 0.05, 1],
    "subsample": [0.5, 0.7, 1],
    "max_features": [0.5, 0.7, 1],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

#Calling RandomizedSearchCV
randomized_cv_gbm_over = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=50, scoring=scorer, cv=5, random_state=1, n_jobs = -1)

#Fitting parameters in RandomizedSearchCV
randomized_cv_gbm_over.fit(X_train_over, y_train_over)

print("Best parameters are {} with CV score={}:" .format(randomized_cv_gbm_over.best_params_,randomized_cv.best_score_))

In [ ]:
tuned_gbm1_over = GradientBoostingClassifier(
    random_state=1,
    subsample=0.5,
    n_estimators=100,
    max_features=1,
    learning_rate=0.01,
)
tuned_gbm1_over.fit(X_train_over, y_train_over)

In [ ]:
# Checking model's performance on training set
gbm1_train_over = model_performance_classification_sklearn(tuned_gbm1_over, X_train_over, y_train_over)
gbm1_train_over

In [ ]:
# Checking model's performance on validation set
gbm1_val_over = model_performance_classification_sklearn(tuned_gbm1_over, X_val, y_val)
gbm1_val_over

## Model Performance Summary and Final Model Selection

In [ ]:
# training performance comparison

models_train_comp_df = pd.concat(
    [
        gbm1_train_orig.T,
        gbm1_train_over.T,
        adb_train_orig.T,
        adb_train_over.T,
        rf_train_orig.T,
        rf_train_over.T
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Gradient boosting trained with Original data",
    "Gradient boosting trained with Oversampled data",
    "AdaBoost trained with Original data",
    "AdaBoost trained with Oversampled data",
    "Random Forest trained with Original data",
    "Random Forest trained with Oversampled data"
]
print("Training performance comparison:")
models_train_comp_df

In [ ]:
# Validation performance comparison

models_train_comp_df = pd.concat(
    [
        gbm1_val_orig.T,
        gbm1_val_over.T,
        adb_val_orig.T,
        adb_val_over.T,
        rf_val_orig.T,
        rf_val_over.T
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Gradient boosting trained with Original data",
    "Gradient boosting trained with Oversampled data",
    "AdaBoost trained with Original data",
    "AdaBoost trained with Oversampled data",
    "Random Forest trained with Original data",
    "Random Forest trained with Oversampled data"
]
print("Validation performance comparison:")
models_train_comp_df

In [ ]:
# Performance on test set
rf_test = model_performance_classification_sklearn(tuned_rf_model, X_test, y_test)
rf_test

# **Observations:**


The model that can be selected is:

**Tuned Random Forest using Original Data**

This model has been selected as it has an overall better F1 score - keeping the Recall, Precision and Accuracy scores at a reasonably better level.

In [ ]:
# Finding important features

feature_names = X_train.columns
importances = tuned_rf_model.feature_importances_

indices = np.argsort(importances)

plt.figure(figsize=(12, 12))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

# **Observations:**

High school education is the parameter of highest importance, followed by Prevailing wage, Job experience and then Master's degree.

## Actionable Insights and Business Recommendations

# **Insights from Exploratory Data Analysis**

66% of the visa requests are from Asia. Followed by Europe and North America with 14.6 and 12.9 respectively.

The dataset given is decently balanced with considerable weightage of approved and rejected cases.

Europe has the highest visa acceptance rate at around 80%, followed by Africa and Asia have an acceptance rate of 70 and 60% respectively

40% of the candidates have a Bachelors degree and 38% of the candidates have a Masters degree - which is the majority of the candidates in the dataset. Very low number of candidates applying have a Doctorate degree or just completed high school.

Most of the candidates (88%) don't require job training, which is interesting because 41.9% of the applicants don't have work experience.

89.4% of the candidates have a full-time position.

Candidates paid hourly are least likely to get their visa accepted.
Yearly wage candidates are most likely to be approved with ~75% approval rate. Weekly and monthly paid candidates also have a 60-65% acceptance rate

Northeast, South, and West receive most of the visa requests - and the pool is mainly Bachelor's and Master's degree holders.

Almost 67% (2/3) of the visa applications are accepted.

Companies with total number of employees between 0-6000 mostly are requesting for a visa

Candidates with no experience have higher wages when they have High School and Master's degrees.
Candidates with Doctorate have the lowest prevailing wages, which is surprising
Candidates with Master's degree have the Highest prevailing wage

Based on EDA, a suitable candidate will have higher education (at least a bachelor's), with job experience, a full-time offer with a yearly, monthly or weekly wage, applying in Northeast, South or West regions, and be from Europe, Africa or Asia.

--------------------------------------------------------------------------------
# **Model Building and Model Selection Conclusions and Business Recommendations**

Model building and selection has been focussed on F1 score as a balanced solution for visa approval. In this use case,  False negatives and false positives both have high cost.

The model that can be selected is: Tuned Random Forest using Original Data. It has an F1 score of 0.84 and 0.83 on train and test sets respectively.
This model also resulted in F1 score of 0.82 on the test set.

This model has been selected as it has an overall better F1 score - keeping the Recall, Precision and Accuracy scores at a reasonabe level.

Based on the selected model, High school education is the parameter of highest importance, followed by Prevailing wage, Job experience and then Master's degree.

Based on visa approval rates and important features of our model, OFLC could screen applications through different levels.
First, sort through applicants by education. (first high school, then masters, bachelors, and end with Doctorate).

Then, order applications based on prevailing wage. The higher the prevailing wage, more likely the visa approval.

Then, classify applications based on job experience. Applicants with job experience are more likely to get their visa accepted.

For further analysis, visa applications can also be analysed based on job sector, experience level and age.

Another aspect that could have an impact on visa applications, is the age of the applicant. It could be a beneficial addition to the data instead of year of establishment.

<font size=6 color='blue'>Power Ahead</font>
___